In [1]:
"""
================================================================================
Calculates mutually exclusive distances for mangrove pixels based on the 1984 baseline:
- If 1984 state == Water (1): Computes Dist_to_land, sets Dist_to_sea to NaN.
- If 1984 state == Land (0): Computes Dist_to_sea, sets Dist_to_land to NaN.
================================================================================
"""

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import Resampling, reproject, transform as transform_coords
from rasterio.transform import xy
from scipy.ndimage import distance_transform_edt
import geopandas as gpd

# Ignore warnings for clean console output
warnings.filterwarnings('ignore', category=RuntimeWarning)

# ==========================================
# STEP 1: FOLDERS AND PATHS CONFIGURATION
# ==========================================
SHP_VAL_PATH = r"D:\MANGROVES\Research_Article_Clean\Created_Data\gmwv4_mask_anthropogenic_validation.shp"
RASTER_STACK_DIR = Path(r"D:\MANGROVES\GMW\V4\gmw_mng_ext_v4112_stack\reprojected_54034")
JRC_BASELINE_DIR = Path(r"D:\MANGROVES\Research_Article_Clean\Created_Data\GSW_1984_baseline")
OUTPUT_CSV_DIR = Path(r"DD:\MANGROVES\Research_Article_Clean\Created_Data\Distance_CSVs")
OUTPUT_CSV_DIR.mkdir(parents=True, exist_ok=True)

# Mapping target benchmark years to their corresponding band index in the GMW stack
TARGET_YEARS = {
    1985: 1,
    1995: 11,
    2005: 21,
    2015: 31,
    2025: 41
}

# ==========================================
# STEP 2: LOAD SHAPEFILE & FILTER PRISTINE TILES
# ==========================================
print("⏳ Loading the validation shapefile...")
gdf_val = gpd.read_file(SHP_VAL_PATH)

if "mask_artif" in gdf_val.columns:
    gdf_pristine = gdf_val[gdf_val["mask_artif"] == 0]
    print(f"🌿 Filtered pristine tiles (mask_artif == 0): {len(gdf_pristine):,}")
else:
    raise KeyError("The column 'mask_artif' was not found in the validation shapefile.")

all_tiles_set = set(gdf_pristine["tile_id"].dropna().astype(str))
all_tiles_set.discard("")

print(f"🎯 Total number of unique pristine tiles to process: {len(all_tiles_set)}")

# ==========================================
# STEP 3: LOOP THROUGH EACH TILE (1-to-1 MATCH)
# ==========================================
for i, tile_id in enumerate(all_tiles_set, 1):
    output_csv = OUTPUT_CSV_DIR / f"mangrove_distances_yearly_{tile_id}.csv"
    
    gmw_path = RASTER_STACK_DIR / f"{tile_id}.tif"
    jrc_path = JRC_BASELINE_DIR / f"{tile_id}_baseline_water.tif"

    if not gmw_path.exists():
        matches = list(RASTER_STACK_DIR.glob(f"*{tile_id}*.tif"))
        if matches:
            gmw_path = matches[0]
        else:
            print(f"[{i}/{len(all_tiles_set)}] ⚠️ GMW stack not found for tile {tile_id}. Skipping.")
            continue

    if not jrc_path.exists():
        jrc_matches = list(JRC_BASELINE_DIR.glob(f"*{tile_id}*.tif"))
        if jrc_matches:
            jrc_path = jrc_matches[0]
        else:
            print(f"[{i}/{len(all_tiles_set)}] ⚠️ JRC baseline raster not found for tile {tile_id}. Skipping.")
            continue

    try:
        with rasterio.open(gmw_path) as src_gmw, rasterio.open(jrc_path) as src_jrc:
            if src_gmw.count < 41:
                print(f"[{i}/{len(all_tiles_set)}] ⚠️ Tile {tile_id} has fewer than 41 bands. Skipping.")
                continue

            gmw_transform = src_gmw.transform
            gmw_crs = src_gmw.crs
            gmw_shape = (src_gmw.height, src_gmw.width)

            pixel_width = abs(src_jrc.transform[0])
            jrc_data = src_jrc.read(1)
            jrc_transform = src_jrc.transform
            jrc_crs = src_jrc.crs

            # 1. Aligner la baseline JRC 1984 sur la grille GMW exacte (Resampling.nearest pour préserver 0 et 1)
            jrc_aligned = np.empty(gmw_shape, dtype=np.uint8)
            reproject(
                source=jrc_data.astype(np.uint8),
                destination=jrc_aligned,
                src_transform=jrc_transform,
                src_crs=jrc_crs,
                dst_transform=gmw_transform,
                dst_crs=gmw_crs,
                resampling=Resampling.nearest
            )

            # 2. Calcul des distances euclidiennes sur toute la tuile JRC
            water_mask = (jrc_data == 1) # 1 = Permanent water
            land_mask = (jrc_data == 0)  # 0 = Land

            dist_to_land = distance_transform_edt(~land_mask) * pixel_width
            dist_to_sea = distance_transform_edt(~water_mask) * pixel_width

            # 3. Alignement des cartes de distance sur la grille GMW
            dist_land_aligned = np.empty(gmw_shape, dtype=np.float32)
            reproject(
                source=dist_to_land.astype(np.float32),
                destination=dist_land_aligned,
                src_transform=jrc_transform,
                src_crs=jrc_crs,
                dst_transform=gmw_transform,
                dst_crs=gmw_crs,
                resampling=Resampling.bilinear
            )

            dist_sea_aligned = np.empty(gmw_shape, dtype=np.float32)
            reproject(
                source=dist_to_sea.astype(np.float32),
                destination=dist_sea_aligned,
                src_transform=jrc_transform,
                src_crs=jrc_crs,
                dst_transform=gmw_transform,
                dst_crs=gmw_crs,
                resampling=Resampling.bilinear
            )

            # 4. Extraction et application de la règle conditionnelle 1984
            yearly_records = []
            for year, band_idx in TARGET_YEARS.items():
                band_data = src_gmw.read(band_idx)
                mng_mask = (band_data == 1)

                if not np.any(mng_mask):
                    continue 

                rows, cols = np.where(mng_mask)
                xs, ys = xy(gmw_transform, rows, cols)
                lons, lats = transform_coords(gmw_crs, 'EPSG:4326', xs, ys)

                # Récupérer l'état 1984 de chaque pixel de mangrove (0 = Terre, 1 = Eau)
                state_1984 = jrc_aligned[mng_mask]

                raw_lands = dist_land_aligned[mng_mask]
                raw_seas = dist_sea_aligned[mng_mask]

                # Application de la règle conditionnelle
                lands = np.full(state_1984.shape, np.nan, dtype=np.float32)
                seas = np.full(state_1984.shape, np.nan, dtype=np.float32)

                # Si eau en 1984 (== 1) -> distance à la terre, sinon NaN pour la terre
                water_in_1984 = (state_1984 == 1)
                lands[water_in_1984] = raw_lands[water_in_1984]

                # Si terre en 1984 (== 0) -> distance à la mer, sinon NaN pour la mer
                land_in_1984 = (state_1984 == 0)
                seas[land_in_1984] = raw_seas[land_in_1984]

                df_year = pd.DataFrame({
                    "tile_id": tile_id,
                    "year": year,
                    "longitude": lons,
                    "latitude": lats,
                    "Dist_to_land": lands,
                    "Dist_to_sea": seas
                })
                yearly_records.append(df_year)

            if yearly_records:
                df_tile_all = pd.concat(yearly_records, ignore_index=True)
                df_tile_all = df_tile_all[["tile_id", "year", "longitude", "latitude", "Dist_to_land", "Dist_to_sea"]]
                df_tile_all.to_csv(output_csv, index=False)
                print(f"[{i}/{len(all_tiles_set)}] ✅ Pristine Tile {tile_id}: Saved conditional CSV ({len(df_tile_all):,} records).")

    except Exception as e:
        print(f"[{i}/{len(all_tiles_set)}] ⚠️ Error processing tile {tile_id}: {e}")
        continue

print(f"\n🎉 Process complete! All conditional yearly distance CSVs are saved in: {OUTPUT_CSV_DIR}")

⏳ Loading the validation shapefile...
🌿 Filtered pristine tiles (mask_artif == 0): 1,261
🎯 Total number of unique pristine tiles to process: 1261
[1/1261] ✅ Pristine Tile S18E035: Saved conditional CSV (398,416 records).
[2/1261] ✅ Pristine Tile S22E167: Saved conditional CSV (11,168 records).
[3/1261] ✅ Pristine Tile N08E093: Saved conditional CSV (30,296 records).
[4/1261] ✅ Pristine Tile N24W098: Saved conditional CSV (50,708 records).
[5/1261] ✅ Pristine Tile N08E098: Saved conditional CSV (197,667 records).
[6/1261] ✅ Pristine Tile S07E156: Saved conditional CSV (65,982 records).
[7/1261] ✅ Pristine Tile S03E130: Saved conditional CSV (381,520 records).
[8/1261] ✅ Pristine Tile N09E080: Saved conditional CSV (12,711 records).
[9/1261] ✅ Pristine Tile N11W067: Saved conditional CSV (48,511 records).
[10/1261] ✅ Pristine Tile S32E133: Saved conditional CSV (189,586 records).
[11/1261] ✅ Pristine Tile N27E055: Saved conditional CSV (358,349 records).
[12/1261] ✅ Pristine Tile N22W160